In [1]:

import pandas as pd

df = pd.read_csv("Creditcard_data.csv")

print("Shape of dataset:", df.shape)
display(df.head())
print("\nClass distribution:")
print(df['Class'].value_counts())


Shape of dataset: (772, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,1
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0



Class distribution:
Class
0    763
1      9
Name: count, dtype: int64


In [2]:
from sklearn.utils import resample

# Separate majority and minority classes
df_majority = df[df.Class == 0]
df_minority = df[df.Class == 1]

# Oversample minority class
df_minority_oversampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

# Combine balanced dataset
df_balanced = pd.concat([df_majority, df_minority_oversampled])

print("Balanced class distribution:")
print(df_balanced['Class'].value_counts())


Balanced class distribution:
Class
0    763
1    763
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

samples = {}

for i in range(1, 6):
    samples[f"Sampling{i}"], _ = train_test_split(
        df_balanced,
        train_size=0.8,
        random_state=42 + i,
        stratify=df_balanced['Class']
    )
    print(f"Sampling{i} shape:", samples[f"Sampling{i}"].shape)


Sampling1 shape: (1220, 31)
Sampling2 shape: (1220, 31)
Sampling3 shape: (1220, 31)
Sampling4 shape: (1220, 31)
Sampling5 shape: (1220, 31)


In [4]:
X_samples = {}
y_samples = {}

for key, sample in samples.items():
    X_samples[key] = sample.drop('Class', axis=1)
    y_samples[key] = sample['Class']


In [5]:
#systematic sampling
def systematic_sampling(X, y, k=2):
    return X.iloc[::k], y.iloc[::k]

X_sys, y_sys = systematic_sampling(X_samples['Sampling2'], y_samples['Sampling2'])


In [6]:
#bootstrap sampling
from sklearn.utils import resample

X_boot, y_boot = resample(
    X_samples['Sampling4'],
    y_samples['Sampling4'],
    replace=True,
    n_samples=len(X_samples['Sampling4']),
    random_state=42
)


In [7]:
#cross-validation sampling
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


In [9]:
models = {
    "M1": LogisticRegression(max_iter=1000),
    "M2": KNeighborsClassifier(n_neighbors=5),
    "M3": DecisionTreeClassifier(random_state=42),
    "M4": RandomForestClassifier(random_state=42),
    "M5": SVC()
}


In [10]:
results = {}

for samp_key in X_samples:
    X = X_samples[samp_key]
    y = y_samples[samp_key]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    results[samp_key] = {}

    for model_key, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results[samp_key][model_key] = round(acc * 100, 2)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [11]:
import pandas as pd

accuracy_table = pd.DataFrame(results).T
accuracy_table


,M1,M2,M3,M4,M5
Sampling1,92.62,97.54,99.59,100.0,72.13
Sampling2,93.03,96.72,99.18,100.0,71.31
Sampling3,91.39,96.72,100.00,100.0,75.41
Sampling4,92.62,97.95,99.59,100.0,70.90
Sampling5,93.03,97.95,99.18,100.0,66.80


In [12]:
print("Accuracy Table:")
display(accuracy_table)

print("\nAverage Accuracy for each Model:")
display(accuracy_table.mean().to_frame(name='Average Accuracy'))

Accuracy Table:


,M1,M2,M3,M4,M5
Sampling1,92.62,97.54,99.59,100.0,72.13
Sampling2,93.03,96.72,99.18,100.0,71.31
Sampling3,91.39,96.72,100.00,100.0,75.41
Sampling4,92.62,97.95,99.59,100.0,70.90
Sampling5,93.03,97.95,99.18,100.0,66.80



Average Accuracy for each Model:


,Average Accuracy
M1,92.538
M2,97.376
M3,99.508
M4,100.000
M5,71.310
